# Разработка A/B-тестирования и анализ результатов

Вы работаете продуктовым аналитиком в компании, которая разрабатывает развлекательное приложение с функцией «бесконечной» ленты, как, например, в приложениях с короткими видео. В вашем приложении существует две модели монетизации: первая — ежемесячная платная подписка, которая позволяет пользователям смотреть ленту без рекламы, вторая — демонстрация рекламы для пользователей, которые ещё не оформили подписку.

Команда разработчиков рекомендательных систем создала новый алгоритм рекомендаций, который, по их мнению, будет показывать более интересный контент для каждого пользователя. Вас, как аналитика, просят помочь рассчитать параметры A/B-теста, который позволит проверить эту гипотезу, и проанализировать его результаты.

## Описание данных

Вы будете работать с тремя таблицами:

- `sessions_project_history.csv` — таблица с историческими данными по сессиям пользователей на период с 2025-08-11 по 2025-09-23. Путь к файлу: `/datasets/sessions_project_history.csv`.

- `sessions_project_test_part.csv` — таблица с данными за первый день проведения A/B-теста, то есть за 2025-10-14. Путь к файлу: `/datasets/sessions_project_test_part.csv`.

- `sessions_project_test.csv` — таблица с данными за весь период проведения A/B-теста, то есть с 2025-10-14 по 2025-11-02. Путь к файлу: `/datasets/sessions_project_test.csv`.

У этих таблиц почти совпадает структура и содержание колонок, различаются лишь периоды наблюдения.

Поля таблиц `sessions_project_history.csv`, `sessions_project_test.csv`, `sessions_project_test_part.csv`:

- `user_id` — идентификатор пользователя;

- `session_id` — идентификатор сессии в приложении;

- `session_date` — дата сессии;

- `session_start_ts` — дата и время начала сессии;

- `install_date` — дата установки приложения;

- `session_number` — порядковый номер сессии для конкретного пользователя;

- `registration_flag` — является ли пользователь зарегистрированным;

- `page_counter` — количество просмотренных страниц во время сессии;

- `region` — регион пользователя;

- `device` — тип устройства пользователя;

- `test_group` — тестовая группа (в таблице с историческими данными этого столбца нет).


## Что нужно сделать
Ваши задачи: рассчитать параметры теста, оценить корректность его проведения и проанализировать результаты эксперимента.

### 1. Работа с историческими данными (EDA)

#### 1.1. Загрузка исторических данных
На первом этапе поработайте с историческими данными приложения:

- Импортируйте библиотеку pandas.

- Считайте и сохраните в датафрейм `sessions_history` CSV-файл с историческими данными о сессиях пользователей `sessions_project_history.csv`.

Выведите на экран первые пять строк полученного датафрейма.

In [ ]:
#Импортируем библиотеку pandas
import pandas as pd
#Сохраняем csv файл с данными о сессиях пользователей в датафрейм sessions_history
sessions_history = pd.read_csv('/datasets/sessions_project_history.csv')
# Выводим первые 5 строк датафрейма
sessions_history.head(5)

#### 1.2. Знакомство с данными
- Для каждого уникального пользователя `user_id` рассчитайте количество уникальных сессий `session_id`.

- Выведите на экран все данные из таблицы `sessions_history` для одного пользователя с наибольшим количеством сессий. Если таких пользователей несколько, выберите любого из них.

- Изучите таблицу для одного пользователя, чтобы лучше понять логику формирования каждого столбца данных.



In [ ]:
# Считаем количество уникальных сессий для каждого уникального пользователя
session_count = sessions_history.groupby('user_id')['session_id'].nunique().reset_index(name='count_unique_session')

In [ ]:
# Находим пользователя с максимальным количеством сессий
max_session_user_id = session_count.loc[session_count['count_unique_session'].idxmax()]

user_data = sessions_history[sessions_history['user_id']==max_session_user_id['user_id']]
# Выводим таблицу для одного пользователя
print(user_data)

#### 1.3. Анализ числа регистраций
Одна из важнейших метрик продукта — число зарегистрированных пользователей. Используя исторические данные, визуализируйте, как менялось число регистраций в приложении за время его существования.

- Агрегируйте исторические данные и рассчитайте число уникальных пользователей и число зарегистрированных пользователей для каждого дня наблюдения. Для простоты считайте, что у пользователя в течение дня бывает одна сессия максимум и статус регистрации в течение одного дня не может измениться.

- Постройте линейные графики общего числа пользователей и общего числа зарегистрированных пользователей по дням. Отобразите их на одном графике.

- Постройте отдельный линейный график доли зарегистрированных пользователей от всех пользователей по дням.

- На обоих графиках должны быть заголовок, подписанные оси X и Y, сетка и легенда.

In [ ]:
# Рассчитываем число уникальных пользователей и число зарегистрированных пользователей для каждого дня
uniq_id_sum_reg_per_day = sessions_history.groupby('session_date').agg(unique_user=('user_id', 'nunique'),registered_user=('registration_flag','sum')) 

In [ ]:
# Импортируем библиотеку matplotlib для дальнейшей визуализации
import matplotlib.pyplot as plt
plot_data = uniq_id_sum_reg_per_day.reset_index()
# Строим линейные графики общего числа пользователей и зарегистрированных по дням
plt.figure(figsize=(20,5))
plt.plot(plot_data['session_date'], plot_data['unique_user'], 'b-', label='Всего пользователей')
plt.plot(plot_data['session_date'], plot_data['registered_user'], 'c-', label='Зарегистрированные пользователи')
plt.title('Общее число пользователей и число зарегистрированных по дням')
plt.xlabel('Дата')
plt.ylabel('Количество пользователей')
plt.xticks(rotation=45, fontsize=10)
plt.legend()
plt.grid()
plt.show()

По визуализации можно сделать вывод, что число зарегистрированных пользователей сильно отстает от общего числа пользователей.

In [ ]:
# Отдельно визуализируем долю зарегистрированных пользователей
plot_data['registration_share'] = plot_data['registered_user']/plot_data['unique_user']
plt.figure(figsize=(20,5))
plt.plot(plot_data['session_date'],plot_data['registration_share']*100)
plt.title('Доля зарегистрированных пользователей от всех пользователей по дням')
plt.xlabel('Дата')
plt.ylabel('Доля, %')
plt.xticks(rotation=45, fontsize=10)
plt.grid()
plt.show()

Доля зарегистрированных пользователей растет, но, учитывая просадку по общему числу пользователей на предыдущем графике нельзя сказать, что данный рост является положительным показателем. 

#### 1.4. Анализ числа просмотренных страниц
Другая важная метрика продукта — число просмотренных страниц в приложении. Чем больше страниц просмотрено, тем сильнее пользователь увлечён контентом, а значит, выше шансы, что он зарегистрируется и оплатит подписку.

- Найдите количество сессий для каждого значения количества просмотренных страниц. Например: одну страницу просмотрели в 29 160 сессиях, две страницы — в 105 536 сессиях и так далее.

- Постройте столбчатую диаграмму, где по оси X будет число просмотренных страниц, по оси Y — количество сессий.

- На диаграмме должны быть заголовок, подписанные оси X и Y.

In [ ]:
# Находим количество сессий для каждого значения просмотренных страниц и визуализируем его
page_count = sessions_history['page_counter'].value_counts().sort_index() 
plt.figure(figsize=(12, 4))
page_count.plot(kind='bar', color='blue', rot=0)
plt.title('Количество сессий с просмотром страниц')
plt.xlabel('Количество страниц')
plt.ylabel('Количество сессий')
plt.show()

Судя по диаграмме, чаще всего за сессию смотрят 3 страницы контента

#### 1.5. Доля пользователей, просмотревших более четырёх страниц
Продуктовая команда продукта считает, что сессии, в рамках которых пользователь просмотрел 4 и более страниц, говорят об удовлетворённости контентом и алгоритмами рекомендаций. Этот показатель является важной прокси-метрикой для продукта.

- В датафрейме `sessions_history` создайте дополнительный столбец `good_session`. В него войдёт значение `1`, если за одну сессию было просмотрено 4 и более страниц, и значение `0`, если было просмотрено меньше.

- Постройте график со средним значением доли успешных сессий от всех сессий по дням за весь период наблюдения.

In [ ]:
# Создаем столбец good_session и задаем ему тип bool
sessions_history['good_session'] = (sessions_history['page_counter']>=4).astype(bool)
# Рассчитываем среднюю долю успешных сессий по дням
day_stats = sessions_history.groupby('session_date')['good_session'].mean()
# Строим график для наглядной визуализации
plt.figure(figsize=(16,4))
plt.plot(day_stats*100)
plt.title('Доля пользователей, просмотревших более четырех страниц')
plt.xlabel('Дата')
plt.ylabel('Доля, %')
plt.xticks(rotation=45, fontsize=10)
plt.grid()
plt.show()

Как мы видим, доля пользователей, просмотревших более 4 страниц остается в диапазоне между 30.5% и 31%. Важно, чтобы во время A/B тестирования, данная метрика не проседала. 

### 2. Подготовка к тесту
При планировании теста необходимо проделать несколько важных шагов:

- Сформулировать нулевую и альтернативную гипотезы

- Определиться с целевой метрикой.

- Рассчитать необходимый размер выборки.

- Исходя из текущих значений трафика рассчитать необходимую длительность проведения теста.

#### 2.1 Формулировка нулевой и альтернативной гипотез

Перед тем как проводить А/B-тест, необходимо сформулировать нулевую и альтернативную гипотезы. Напомним изначальное условие: команда разработчиков рекомендательных систем создала новый алгоритм, который, по их мнению, будет показывать более интересный контент для каждого пользователя.

О какой метрике идёт речь? Как она будет учтена в формулировке гипотез?

Проанализировав датафрейм sessions_history, мы пришли к выводу, что столбец good_session можно назвать ключевой метрикой, так как данный столбец напрямую отражает суть заинтересованности пользователей в контенте. Чем среднее good_session больше, тем более интересным можно считать контент, который предлагается пользователю

Сформулируйте нулевую и альтернативную гипотезы:






Н0 Нулевая гипотеза: Среднее значение good_session в группе с новым алгоритмом статистически не отличается от контрольной группы

Н1 Альтернативная гипотеза: Среднее значение good_session в группе с новым алгоритмом больше, чем в контрольной группе, что указывает на повышенную заинтересованность контентом

#### 2.2. Расчёт размера выборки
В рамках курса вы уже рассчитывали размеры выборки и  использовали для этого онлайн-калькулятор. В этом задании предлагаем воспользоваться готовым кодом и рассчитать необходимое для вашего эксперимента количество пользователей.

Для этого установите в коде ниже следующие параметры:

- Уровень значимости — 0.05.

- Вероятность ошибки второго рода — 0.2.

- Мощность теста.

- Минимальный детектируемый эффект, или MDE, — 3%. Обратите внимание, что здесь нужно указать десятичную дробь, а не процент.

При расчёте размера выборки используйте метод `solve_power()` из класса `power.NormalIndPower` модуля `statsmodels.stats`.

Запустите ячейку и изучите полученное значение.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Задаём параметры
alpha = 0.05  # Уровень значимости
beta = 0.2  # Ошибка второго рода, часто 1 - мощность
power = 1 - beta  # Мощность теста
p = 0.3 # Базовый уровень доли
mde = 0.03 * p  # Минимальный детектируемый эффект
p1=p
effect_size = proportion_effectsize(p1, p1 + mde)

# Инициализируем класс NormalIndPower
power_analysis = NormalIndPower()

# Рассчитываем размер выборки
sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1 # Равномерное распределение выборок
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

#### 2.3. Расчёт длительности A/B-теста

Используйте данные о количестве пользователей в каждой выборке и среднем количестве пользователей приложения. Рассчитайте длительность теста, разделив одно на другое.

- Рассчитайте среднее количество уникальных пользователей приложения в день.

- Определите длительность теста исходя из рассчитанного значения размера выборок и среднего дневного трафика приложения. Количество дней округлите в большую сторону.

In [ ]:
from math import ceil

# Среднее количество пользователей приложения в день по историческим данным
avg_daily_users = round(sessions_history.groupby('session_date')['user_id'].nunique().mean())

# Рассчитываем длительность теста в днях как отношение размера выборки к среднему числу пользователей
test_duration = ceil(sample_size/avg_daily_users*2)

print(f"Рассчитанная длительность A/B-теста при текущем уровне трафика в {avg_daily_users} пользователей в день составит {test_duration} дней")

### 3. Мониторинг А/В-теста

#### 3.1. Проверка распределения пользователей

A/B-тест успешно запущен, и уже доступны данные за первые три дня. На этом этапе нужно убедиться, что всё идёт хорошо: пользователи разделены правильным образом, а интересующие вас метрики корректно считаются.

- Считайте и сохраните в датафрейм `sessions_test_part` CSV-файл с историческими данными о сессиях пользователей `sessions_project_test_part.csv`.

- Рассчитайте количество уникальных пользователей в каждой из экспериментальных групп для одного дня наблюдения.

- Рассчитайте и выведите на экран процентную разницу в количестве пользователей в группах A и B. Постройте любую удобную визуализацию, на которой будет видно возможное различие двух групп.

Для расчёта процентной разницы воспользуйтесь формулой:
$$P = 100 \cdot  \frac{|A − B|}{A}$$

In [ ]:
# Сохраняем csv файл в датафрейм sessions_test_part
sessions_test_part = pd.read_csv('/datasets/sessions_project_test_part.csv')
# Рассчитаем количество уникальных пользователей в группе A и B
A = sessions_test_part[sessions_test_part['test_group']=='A']['user_id'].nunique()
B = sessions_test_part[sessions_test_part['test_group']=='B']['user_id'].nunique()
# Рассчитаем процентную разницу между группами
P = 100 * (A-B)/A
# Выведем процентную разницу на экран
print(f'Процетная разница равна {P} процента')
# Построим столбчатую диаграмму для визуализации количества пользователей в двух группах
plt.figure(figsize=(12,6))
plt.bar(['Группа A','Группа B'],[A,B])
plt.title('Количество уникальных пользователей по группам')
plt.ylabel('Количество уникальных пользователей')
plt.grid()
plt.show()

Как мы видим по процентной разнице и по диаграмме - различия между группами незначительны

#### 3.2. Проверка пересечений пользователей
Помимо проверки равенства количества пользователей в группах, полезно убедиться в том, что группы независимы. Для этого нужно убедиться, что никто из пользователей случайно не попал в обе группы одновременно.

- Рассчитайте количество пользователей, которые встречаются одновременно в группах A и B, или убедитесь, что таких нет.

In [ ]:
# Посмотрим есть ли пользователи, которые находятся в обеих группах одновременно
A_group = sessions_test_part[sessions_test_part['test_group'] == 'A']['user_id']
B_group = sessions_test_part[sessions_test_part['test_group'] == 'B']['user_id']

intersection = list(set(A_group) & set(B_group))
print(intersection) 

Как мы видим, таких нет

#### 3.3. Равномерность разделения пользователей по устройствам
Полезно также убедиться в том, что пользователи равномерно распределены по всем доступным категориальным переменным — типам устройств и регионам.

Постройте две диаграммы:

- доля каждого типа устройства для пользователей из группы A,

- доля каждого типа устройства для пользователей из группы B.

Постарайтесь добавить на диаграммы все необходимые подписи, пояснения и заголовки, которые позволят сделать вывод о том, совпадает ли распределение устройств в группах A и B.


In [ ]:
# Строим диаграмму для группы A
plt.figure(figsize=(12, 6))
A_shares=(sessions_test_part[sessions_test_part['test_group']=='A'].groupby('device')['user_id'].nunique().sort_values(ascending=False))
A_shares_prop=A_shares/A_shares.sum()*100
A_shares_prop.plot(kind='bar',rot=0)
plt.title('Распределение пользователей по устройствам (Группа A)')
plt.xlabel('Категории устройств')
plt.ylabel('Доля пользователей в %')
plt.tight_layout()
plt.grid()
plt.show()

In [ ]:
# Строим диаграмму для группы B
plt.figure(figsize=(12, 6))
A_shares=(sessions_test_part[sessions_test_part['test_group']=='B'].groupby('device')['user_id'].nunique().sort_values(ascending=False))
A_shares_prop=A_shares/A_shares.sum()*100
A_shares_prop.plot(kind='bar',rot=0)
plt.title('Распределение пользователей по устройствам (Группа B)')
plt.xlabel('Категории устройств')
plt.ylabel('Доля пользователей в %')
plt.tight_layout()
plt.grid()
plt.show()

Исходя из визуализации распределения пользователей по устройствам, можно сказать, что в целом распределение является равномерным

#### 3.4. Равномерность распределения пользователей по регионам
Теперь убедитесь, что пользователи равномерно распределены по регионам.

Постройте две диаграммы:

- доля каждого региона для пользователей из группы A,

- доля каждого региона для пользователей из группы B.

Постарайтесь добавить на диаграммы все необходимые подписи, пояснения и заголовки, которые позволят сделать вывод о том, совпадает ли распределение регионов в группах A и B. Постарайтесь использовать другой тип диаграммы, не тот, что в прошлом задании.


In [ ]:
# Строим диаграмму для группы A
plt.figure(figsize=(12, 6))
A_shares=(sessions_test_part[sessions_test_part['test_group']=='A'].groupby('region')['user_id'].nunique().sort_values(ascending=False))
A_shares_prop=A_shares/A_shares.sum()*100
A_shares_prop.plot(kind='bar',rot=0)
plt.title('Распределение пользователей по регионам (Группа A)')
plt.xlabel('Регион')
plt.ylabel('Доля пользователей в %')
plt.tight_layout()
plt.grid()
plt.show()

In [ ]:
# Строим диаграмму для группы A
plt.figure(figsize=(12, 6))
A_shares=(sessions_test_part[sessions_test_part['test_group']=='B'].groupby('region')['user_id'].nunique().sort_values(ascending=False))
A_shares_prop=A_shares/A_shares.sum()*100
A_shares_prop.plot(kind='bar',rot=0)
plt.title('Распределение пользователей по регионам (Группа B)')
plt.xlabel('Регион')
plt.ylabel('Доля пользователей в %')
plt.tight_layout()
plt.grid()
plt.show()

Как мы видим, распределение по регионам сбалансированное

#### 3.5. Вывод после проверки A/B-теста

На основе проведённого анализа A/B-теста сформулируйте и запишите свои выводы. В выводе обязательно укажите:

- Было ли обнаружено различие в количестве пользователей в двух группах.

- Являются ли выборки независимыми. Было ли обнаружено пересечение пользователей из тестовой и контрольной групп.

- Сохраняется ли равномерное распределение пользователей тестовой и контрольной групп по категориальным переменным: устройствам и регионам.

Сделайте заключение: корректно ли проходит A/B-тест, или наблюдаются какие-либо нарушения.

Вывод: 
1. Различие в количестве пользователей было обнаружено, но оно незначительное - около 0.7%
2. Выборки независимы, пересечение пользователей не обнаружено
3. Распределение по устройствам в целом можно назвать равномерным. Распределение по регионам также сбалансированное

В целом тест проходит корректно, критических нарушений нет.

### 4. Проверка результатов A/B-теста

A/B-тест завершён, и у вас есть результаты за все дни проведения эксперимента. Необходимо убедиться в корректности теста и верно интерпретировать результаты.

#### 4.1. Получение результатов теста и подсчёт основной метрики

- Считайте и сохраните в датафрейм `sessions_test` CSV-файл с историческими данными о сессиях пользователей `sessions_project_test.csv`.

- В датафрейме `sessions_test` создайте дополнительный столбец `good_session`. В него войдёт значение `1`, если за одну сессию было просмотрено 4 и более страниц, и значение `0`, если просмотрено меньше.

In [ ]:
# Сохраняем csv файл в датафрейм sessions_test
sessions_test = pd.read_csv('/datasets/sessions_project_test.csv')
# Создаем столбец good_session 
sessions_test['good_session'] = (sessions_test['page_counter']>=4).astype(bool)
# Отобразим полный датафрейм, чтобы увидеть, что столбец добавился корректно
sessions_test.info()

#### 4.2. Проверка корректности результатов теста

Прежде чем приступать к анализу ключевых продуктовых метрик, необходимо убедиться, что тест проведён корректно и вы будете сравнивать две сопоставимые группы.

- Рассчитайте количество уникальных сессий для каждого дня и обеих тестовых групп, используя группировку.

- Проверьте, что количество уникальных дневных сессий в двух выборках не различается или различия не статистически значимыми. Используйте статистический тест, который позволит сделать вывод о равенстве средних двух выборок.

- В качестве ответа выведите на экран полученное значение p-value и интерпретируйте его.

In [ ]:
# Воспользуемся библиотекой spicy для проведение статистического теста
from scipy import stats
# Поочередно рассчитаем количество уникальных сессий для каждого дня и обеих тестовых групп
A_sessions = sessions_test[sessions_test['test_group']=='A'].groupby('session_date')['session_id'].nunique()
B_sessions = sessions_test[sessions_test['test_group']=='B'].groupby('session_date')['session_id'].nunique()
# Проведем статистический тест (t-тест)
alpha = 0.05
stat_ttest, p_value_ttest = stats.ttest_ind(
          A_sessions,
          B_sessions)
if p_value_ttest>alpha:
    print("Статистически значимых различий нет")
else: 
    print("Присутствуют статистически значимые различия")
    
print(p_value_ttest)

Результаты статистического теста говорят о том, что среднее количество уникальных сессий в день между группами A и B статистически незначительно.

#### 4.3. Сравнение доли успешных сессий

Когда вы убедились, что количество сессий в обеих выборках не различалось, можно переходить к анализу ключевой метрики — доли успешных сессий.

Используйте созданный на первом шаге задания столбец `good_session` и рассчитайте долю успешных сессий для выборок A и B, а также разницу в этом показателе. Полученный вывод отобразите на экране.

In [ ]:
# Рассчитаем долю успешных сессий для группы A
share_A=(sessions_test.loc[sessions_test['test_group']=='A','good_session'].sum()/sessions_test.loc[sessions_test['test_group']=='A','good_session'].count()).mean()
# Рассчитаем долю успешных сессий для группы B
share_B=(sessions_test.loc[sessions_test['test_group']=='B','good_session'].sum()/sessions_test.loc[sessions_test['test_group']=='B','good_session'].count()).mean()
print(f'Доля успешных сессий группы A={share_A}')
print(f'Доля успешных сессий группы B={share_B}')
share_diff = round((share_B - share_A) * 100,2)
print(f'Разница успешных сессий между группами составила {share_diff}%. В группе B показатель оказался выше')


#### 4.4. Насколько статистически значимо изменение ключевой метрики

На предыдущем шаге вы убедились, что количество успешных сессий в тестовой выборке примерно на 1.1% выше, чем в контрольной, но делать выводы только на основе этого значения будет некорректно. Для принятия решения всегда необходимо отвечать на вопрос: является ли это изменение статистически значимым.

- Используя статистический тест, рассчитайте, является ли изменение в метрике доли успешных сессий статистически значимым.

- Выведите на экран полученное значение p-value и свои выводы о статистической значимости. Напомним, что уровень значимости в эксперименте был выбран на уровне 0.05.

Так как в данном контексте идет речь о сравнении долей в двух независимых выборках, мы применим z-тест пропорций

In [ ]:
# Размер выборки A
n_a=sessions_test.loc[sessions_test['test_group']=='A','good_session'].shape[0]
# Размер выборки B
n_b=sessions_test.loc[sessions_test['test_group']=='B','good_session'].shape[0]
# Количество успешных сессий в группе A
m_a=sessions_test.loc[(sessions_test.test_group=='A')&(sessions_test.good_session==1)].shape[0]
# Количество успешных сессий в группе B
m_b=sessions_test.loc[(sessions_test.test_group=='B')&(sessions_test.good_session==1)].shape[0]
# Доля успеха в группе A
p_a=m_a/n_a
# Доля успеха в группе B
p_b=m_b/n_b

if (p_a*n_a > 10)and((1-p_a)*n_a > 10)and(p_b*n_b > 10)and((1-p_b)*n_b > 10):
    print('Предпосылка о достаточном количестве данных выполняется!')
else:
    print('Предпосылка о достаточном количестве данных НЕ выполняется!')
from statsmodels.stats.proportion import proportions_ztest    
alpha=0.05

stat_ztest, p_value_ztest = proportions_ztest(
        [m_a,m_b],
        [n_a,n_b],
        alternative='smaller')

p_value_ztest

if p_value_ztest > alpha:
    print(f'pvalue={p_value_ztest} > {alpha}')
    print('Нулевая гипотеза находит подтверждение!')
else:
    print(f'pvalue={p_value_ztest} < {alpha}')
    print('Нулевая гипотеза не находит подтверждения!')

По итогу статистического теста мы отвергаем нулевую гипотезу и принимаем альтернативную, которая гласила, что новый алгоритм рекомендаций повысил интерес к контенту у пользователей.

#### 4.5. Вывод по результатам A/B-эксперимента

На основе проведённого анализа результатов теста сформулируйте и запишите свои выводы для команды разработки приложения. В выводе обязательно укажите:

- Характеристики проведённого эксперимента, количество задействованных пользователей и длительность эксперимента.

- Повлияло ли внедрение нового алгоритма рекомендаций на рост ключевой метрики и как.

- Каким получилось значение p-value для оценки статистической значимости выявленного эффекта.

- Стоит ли внедрять нововведение в приложение.

Выводы:
1. Необходимый размер выборки для каждой группы составил 41040, длительность эксперимента - 9 дней.
2. Внедрение нового алгоритма рекомендаций положительно сказалось на росте ключевой метрики, а именно - увеличило её на 1.1%
3. pvalue=0.0001574739988036123
4. Данное новводенение стоит внедрения в приложение